# その他の組み込みミドルウェアのテスト

## 1、ModelCallLimitMiddleware ミドルウェア

モデルの呼び出し回数を制限し、無限ループを回避し、呼び出しコストを抑える。

### 例1：セッション全体の制限-正常終了

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

import os
from http.client import responses

from dataclasses_json import config
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# 環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 要約生成用のモデルを初期化
model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,  # 各スレッド最大2回のモデル呼び出し
            # run_limit=5,   # 各実行で最大5回
            exit_behavior="end",  # 制限に達したら終了
        ),
    ],
)

config = {"configurable": {"thread_id": "1"}}

print("=" * 30, "> first <", "=" * 30)
response_first = agent.invoke({
    "messages": [HumanMessage("こんにちは")]},
    config=config
)

for msg in response_first["messages"]:
    msg.pretty_print()

print("=" * 30, "> second <", "=" * 30)
response_second = agent.invoke({
    "messages": [HumanMessage("あなたは誰ですか？")]},
    config=config
)

for msg in response_second["messages"]:
    msg.pretty_print()

print("=" * 30, "> third <", "=" * 30)
response_third = agent.invoke({
    "messages": [HumanMessage("何を手伝ってもらえますか？")]},
    config=config
)
for msg in response_third["messages"]:
    msg.pretty_print()

============================== > first < ==============================

================================ Human Message =================================

こんにちは
================================== Ai Message ==================================

こんにちは！どうされましたか？


============================== > second < ==============================

================================ Human Message =================================

こんにちは
================================== Ai Message ==================================

こんにちは！どうされましたか？
================================ Human Message =================================

あなたは誰ですか？
================================== Ai Message ==================================

私はOpenAIが開発したAIアシスタントです。  
質問に答えたり、文章を一緒に考えたり、調べものの整理を手伝ったりできます。


============================== > third < ==============================

================================ Human Message =================================

こんにちは
================================== Ai Message ==================================

こんにちは！どうされましたか？
================================ Human Message =================================

あなたは誰ですか？
================================== Ai Message ==================================

私はOpenAIが開発したAIアシスタントです。  
質問に答えたり、文章を一緒に考えたり、調べものの整理を手伝ったりできます。
================================ Human Message =================================

何を手伝ってもらえますか？
================================== Ai Message ==================================

Model call limits exceeded: thread limit (2/2)


### 例2：セッション全体の制限-例外をスロー

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

from typing import List

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2,
            # run_limit=5,
            exit_behavior="error",
        ),
    ],
)

config = {"configurable": {"thread_id": "1"}}

print("=" * 30, "> first <", "=" * 30)
response_first = agent.invoke({
    "messages": [HumanMessage("こんにちは")]},
    config=config
)

for msg in response_first["messages"]:
    msg.pretty_print()

print("=" * 30, "> second <", "=" * 30)
response_second = agent.invoke({
    "messages": [HumanMessage("あなたは誰ですか？")]},
    config=config
)

for msg in response_second["messages"]:
    msg.pretty_print()

print("=" * 30, "> third <", "=" * 30)
response_third = agent.invoke({
    "messages": [HumanMessage("何を手伝ってもらえますか？")]},
    config=config
)
for msg in response_third["messages"]:
    msg.pretty_print()

============================== > first < ==============================

================================ Human Message =================================

こんにちは
================================== Ai Message ==================================

こんにちは！どうされましたか？


============================== > second < ==============================

================================ Human Message =================================

こんにちは
================================== Ai Message ==================================

こんにちは！どうされましたか？
================================ Human Message =================================

あなたは誰ですか？
================================== Ai Message ==================================

私はAIアシスタントです。  
質問に答えたり、文章を作ったり、考えを整理するお手伝いができます。


============================== > third < ==============================

ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)

### 例3：単回呼び出し制限-正常終了

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek

from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv

load_dotenv()
model = ChatDeepSeek(
    model="any",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>")
)


class ContactInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザー氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの携帯電話番号")


class EventInfo(BaseModel):
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント発生日")


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            # thread_limit=2,
            run_limit=3,
            exit_behavior="end",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]
)

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]},
    config=config
)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

こんにちは
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: 田中
    email: songhongkang@atguigu.cn
    phone: 12345678912
  EventInfo (call_2)
 Call ID: call_2
  Args:
    event_name: プロジェクトキックオフミーティング
    date: 2026-03-27
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)

### 例4：単回呼び出し制限-例外をスロー

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek

from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv

load_dotenv(override=True)
model = ChatDeepSeek(
    model="any",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>")
)


class ContactInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザー氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの携帯電話番号")


class EventInfo(BaseModel):
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント発生日")


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ModelCallLimitMiddleware(
            # thread_limit=2,
            run_limit=3,
            exit_behavior="error",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]
)

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]},
    config=config
)

for msg in response["messages"]:
    msg.pretty_print()


ModelCallLimitExceededError: Model call limits exceeded: run limit (3/3)

## 2、ToolCallLimitMiddleware ミドルウェア

ツール呼び出し回数を制限する。**すべてのツール**の呼び出し総数を制限することも、**特定のツール**の呼び出し回数を制限することもできる。

**用途：**

- コストの高い外部APIの過剰な呼び出しを回避
- Webクローラーやデータベースクエリのリクエスト数を制限
- Agentが無限ループに陥るのを回避

**終了動作には3つのモードがある：**

- **error**：直接例外をスロー
- **end**：セッション全体を終了
- **continue**：Agentの実行を継続する。これがデフォルトの動作で、この場合Agentはツール呼び出しが制限を超えた情報をモデルに渡し、モデルが自主的に以降の動作を決定する。モデルの能力が不十分な場合、無限ループに陥る可能性がある。これを避けるため、私たちが実装した fake server は20%の確率で正しいレスポンスを出力し、ループを終了できるようにしている。

### 例1：セッション全体の制限-正常終了

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek

from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv

load_dotenv(override=True)
model = ChatDeepSeek(
    model="any",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>")
)


class ContactInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザー氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの携帯電話番号")


class EventInfo(BaseModel):
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント発生日")


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ToolCallLimitMiddleware(
            # thread_limit=2,  # 各スレッド最大2回のツール呼び出し
            run_limit=2,  # 各実行で最大2回
            exit_behavior="end",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]
)

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]},
    config=config
)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

こんにちは
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: 田中
    email: songhongkang@atguigu.cn
    phone: 12345678912
  EventInfo (call_2)
 Call ID: call_2
  Args:
    event_name: プロジェクトキックオフミーティング
    date: 2026-03-27
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)

### 例2：セッション全体の制限-例外をスロー

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek

from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv

load_dotenv(override=True)
model = ChatDeepSeek(
    model="any",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>")
)


class ContactInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザー氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの携帯電話番号")


class EventInfo(BaseModel):
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント発生日")


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ToolCallLimitMiddleware(
            # thread_limit=2,
            run_limit=2,
            exit_behavior="error",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]
)

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]},
    config=config
)

for msg in response["messages"]:
    msg.pretty_print()


ToolCallLimitExceededError: Tool call limit reached: run limit exceeded (4/2 calls).

### 例3：単回呼び出し制限-継続実行

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek

from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv

load_dotenv(override=True)
model = ChatDeepSeek(
    model="any",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>")
)


class ContactInfo(BaseModel):
    """ユーザーの連絡先情報"""
    name: str = Field(description="ユーザー氏名")
    email: str = Field(description="ユーザーのメールアドレス")
    phone: str = Field(description="ユーザーの携帯電話番号")


class EventInfo(BaseModel):
    event_name: str = Field(description="イベント名")
    date: str = Field(description="イベント発生日")


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  # Required for thread limiting
    tools=[],
    middleware=[
        ToolCallLimitMiddleware(
            # thread_limit=2,
            run_limit=2,
            exit_behavior="continue",
        ),
    ],
    response_format=Union[ContactInfo, EventInfo]
)

config = {"configurable": {"thread_id": "1"}}

# seen = set()

response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]},
    config=config
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

こんにちは
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: 田中
    email: songhongkang@atguigu.cn
    phone: 12345678912
  EventInfo (call_2)
 Call ID: call_2
  Args:
    event_name: プロジェクトキックオフミーティング
    date: 2026-03-27
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)

## 3、ModelFallbackMiddleware ミドルウェア

フェイルオーバー用。メインモデルにアクセスできない場合、バックアップモデルを有効にする。

### 例

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv(override=True)

agent = create_agent(
    model="deepseek:fake_model",
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            "deepseek-v4-flash",
            "deepseek-v4-pro",
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("あなたは誰ですか？")]
})

last_msg = response["messages"][-1]
print(last_msg)

print('=' * 30, '-> model_name <-', '=' * 30)
print(last_msg.response_metadata.get("model_name"))

content='私はDeepSeekです。DeepSeekという中国の会社が開発したAIアシスタントです。無料でご利用いただけます。テキストでの会話や質問応答、ファイルのアップロード（画像、PDF、Word、Excel、PPTなど）による内容の読み取りも可能です。また、Web検索機能もありますが、ユーザーが手動でスイッチをオンにする必要があります。\n\n私は最新バージョンのDeepSeekモデルを使用しており、知識カットオフ日は2025年5月です。何かお手伝いできることがあれば、どうぞお気軽にご質問ください！' additional_kwargs={'refusal': None, 'reasoning_content': 'The user asks "Who are you?" in Japanese. This is a simple self-identification question. I should respond in Japanese since that\'s the language used. Need to give a clear, friendly introduction of myself as DeepSeek AI. Keep it concise and helpful. Mention my capabilities and willingness to assist.'} response_metadata={'token_usage': {'completion_tokens': 208, 'prompt_tokens': 88, 'total_tokens': 296, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 62, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens':

## 4、LLMToolSelectorMiddleware ミドルウェア

インテリジェントなツール選別。

ツールが多すぎる場合、**サブモデル**を使って最も関連性の高いいくつかのツールを選別する。

パラメータ：

- `model`：ツール選別に使用するサブモデル
- `max_tools`：呼び出し可能なツールの総数を制限
- `always_include`：指定したツールはカウントされない

例えば：

In [7]:
from langchain.agents.middleware import LLMToolSelectorMiddleware

tool_selector = LLMToolSelectorMiddleware(
    model="openai:gpt-5.4-mini",
    max_tools=5,  # 最大5つのツールを選択
    always_include=["get_weather"]
)

agent = create_agent(
    model="deepseek-v4-flash",
    # tools=[...100個のツール...],  # 大量のツール
    middleware=[tool_selector]
)

### 例1

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .env ファイルから環境変数を読み込む
load_dotenv(override=True)

model_out = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE")
)

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .env ファイルから環境変数を読み込む
load_dotenv(override=True)

model_in = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE")
)

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    return f"{city}は今日晴れです"


@tool
def get_news():
    """本日の国内ニュース概要を照会する"""
    return ("本日の国内ニュース概要："
            "中国のタンカー3隻がホルムズ海峡を通過")


@tool
def calculate(num1: int, num2: int) -> int:
    """
    数学計算を実行する

    Args:
        num1: 1つ目の加数
        num2: 2つ目の加数
    """
    return num1 + num2


@tool
def search_stock(symbol: str):
    """
    株価情報を照会する

    Args:
        symbol: 銘柄コード
    """
    return "この銘柄は今日好調です"


agent = create_agent(
    model=model_out,
    tools=[get_weather, get_news, calculate, search_stock],
    middleware=[
        LLMToolSelectorMiddleware(
            model=model_in,
            max_tools=0,
            always_include=["get_weather"]
        ),
    ],
)

response = agent.invoke({
    "messages": HumanMessage("北京の今日の天気はどうですか？今日のニュース概要も教えてください")
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

北京の今日の天気はどうですか？今日のニュース概要も教えてください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_cOplA3xOLZ42ZhgDx5NU14TS)
 Call ID: call_cOplA3xOLZ42ZhgDx5NU14TS
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京は今日晴れです
================================== Ai Message ==================================

北京の今日の天気は晴れです。

今日のニュース概要については、こちらの環境では最新ニュースの取得機能がないため、正確な当日ニュースを直接確認できません。  
必要なら、**北京関連のニュースを要約するための観点**や、**見出しを貼っていただければ要約**はできます。


### 例2

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    return f"{city}は今日晴れです"


@tool
def get_news():
    """本日の国内ニュース概要を照会する"""
    return ("本日の国内ニュース概要："
            "中国のタンカー3隻がホルムズ海峡を通過")


@tool
def calculate(num1: int, num2: int) -> int:
    """
    数学計算を実行する

    Args:
        num1: 1つ目の加数
        num2: 2つ目の加数
    """
    return num1 + num2


@tool
def search_stock(symbol: str):
    """
    株価情報を照会する

    Args:
        symbol: 銘柄コード
    """
    return "この銘柄は今日好調です"


agent = create_agent(
    model=model_out,
    tools=[get_weather, get_news, calculate, search_stock],
    middleware=[
        LLMToolSelectorMiddleware(
            model=model_in,
            max_tools=0,
            always_include=["get_news"]
        ),
    ],
)

response = agent.invoke({
    "messages": HumanMessage("北京の今日の天気はどうですか？今日のニュース概要も教えてください")
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

北京の今日の天気はどうですか？今日のニュース概要も教えてください
================================== Ai Message ==================================
Tool Calls:
  get_news (call_a25zPvc88SkmZZxYGJqLGbji)
 Call ID: call_a25zPvc88SkmZZxYGJqLGbji
  Args:
================================= Tool Message =================================
Name: get_news

本日の国内ニュース概要：中国のタンカー3隻がホルムズ海峡を通過
================================== Ai Message ==================================

北京の今日の天気は、こちらではリアルタイム取得できません。  
ただし、今日のニュース概要は以下です。

- **本日の国内ニュース概要：中国のタンカー3隻がホルムズ海峡を通過**

北京の天気も含めて知りたい場合は、天気情報を取得できる環境があれば案内できます。必要なら「最高気温・最低気温・降水確率」までわかる形で調べる手順もお伝えします。


### 例3

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    return f"{city}は今日晴れです"


@tool
def get_news():
    """本日の国内ニュース概要を照会する"""
    return ("本日の国内ニュース概要："
            "中国のタンカー3隻がホルムズ海峡を通過")


@tool
def calculate(num1: int, num2: int) -> int:
    """
    数学計算を実行する

    Args:
        num1: 1つ目の加数
        num2: 2つ目の加数
    """
    return num1 + num2


@tool
def search_stock(symbol: str):
    """
    株価情報を照会する

    Args:
        symbol: 銘柄コード
    """
    return "この銘柄は今日好調です"


agent = create_agent(
    model=model_out,
    tools=[get_weather, get_news, calculate, search_stock],
    middleware=[
        LLMToolSelectorMiddleware(
            model=model_in,
            max_tools=0,
            always_include=["get_weather", "get_news"]
        ),
    ],
)

response = agent.invoke({
    "messages": HumanMessage("北京の今日の天気はどうですか？今日のニュース概要も教えてください")
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

北京の今日の天気はどうですか？今日のニュース概要も教えてください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_nu7OpJKqnXHwfh9tAhq5dclS)
 Call ID: call_nu7OpJKqnXHwfh9tAhq5dclS
  Args:
    city: 北京
  get_news (call_C8KYkUCvrwXb32v2ujkdalT1)
 Call ID: call_C8KYkUCvrwXb32v2ujkdalT1
  Args:
================================= Tool Message =================================
Name: get_weather

北京は今日晴れです
================================= Tool Message =================================
Name: get_news

本日の国内ニュース概要：中国のタンカー3隻がホルムズ海峡を通過
================================== Ai Message ==================================

北京の今日の天気は**晴れ**です。

今日のニュース概要は、**中国のタンカー3隻がホルムズ海峡を通過**した、という内容です。


### 例4

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    return f"{city}は今日晴れです"


@tool
def get_news():
    """本日の国内ニュース概要を照会する"""
    return ("本日の国内ニュース概要："
            "中国のタンカー3隻がホルムズ海峡を通過")


@tool
def calculate(num1: int, num2: int) -> int:
    """
    数学計算を実行する

    Args:
        num1: 1つ目の加数
        num2: 2つ目の加数
    """
    return num1 + num2


@tool
def search_stock(symbol: str):
    """
    株価情報を照会する

    Args:
        symbol: 銘柄コード
    """
    return "この銘柄は今日好調です"


agent = create_agent(
    model=model_out,
    tools=[get_weather, get_news, calculate, search_stock],
    middleware=[
        LLMToolSelectorMiddleware(
            model=model_in,
            max_tools=1,
            always_include=["get_weather"]
        ),
    ],
)

response = agent.invoke({
    "messages": HumanMessage("北京の今日の天気はどうですか？今日のニュース概要も教えてください")
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

北京の今日の天気はどうですか？今日のニュース概要も教えてください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_NHQBZl6pnki2svqdJqxihc3I)
 Call ID: call_NHQBZl6pnki2svqdJqxihc3I
  Args:
    city: 北京
  get_news (call_7BJ3lxYcZ3aNPxFpRcsbBVTw)
 Call ID: call_7BJ3lxYcZ3aNPxFpRcsbBVTw
  Args:
================================= Tool Message =================================
Name: get_weather

北京は今日晴れです
================================= Tool Message =================================
Name: get_news

本日の国内ニュース概要：中国のタンカー3隻がホルムズ海峡を通過
================================== Ai Message ==================================

北京の今日の天気は**晴れ**です。  
今日のニュース概要は、**「中国のタンカー3隻がホルムズ海峡を通過」**です。


### 例5

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    return f"{city}は今日晴れです"


@tool
def get_news():
    """本日の国内ニュース概要を照会する"""
    return ("本日の国内ニュース概要："
            "中国のタンカー3隻がホルムズ海峡を通過")


@tool
def calculate(num1: int, num2: int) -> int:
    """
    数学計算を実行する

    Args:
        num1: 1つ目の加数
        num2: 2つ目の加数
    """
    return num1 + num2


@tool
def search_stock(symbol: str):
    """
    株価情報を照会する

    Args:
        symbol: 銘柄コード
    """
    return "この銘柄は今日好調です"


agent = create_agent(
    model=model_out,
    tools=[get_weather, get_news, calculate, search_stock],
    middleware=[
        LLMToolSelectorMiddleware(
            model=model_in,
            max_tools=1,
            always_include=["get_news"]
        ),
    ],
)

response = agent.invoke({
    "messages": HumanMessage("北京の今日の天気はどうですか？今日のニュース概要も教えてください")
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

北京の今日の天気はどうですか？今日のニュース概要も教えてください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_XewJDpuIhAKi9b1ZsK94jowv)
 Call ID: call_XewJDpuIhAKi9b1ZsK94jowv
  Args:
    city: 北京
  get_news (call_heLRj35QnlalZmHwOxP2oCbb)
 Call ID: call_heLRj35QnlalZmHwOxP2oCbb
  Args:
================================= Tool Message =================================
Name: get_weather

北京は今日晴れです
================================= Tool Message =================================
Name: get_news

本日の国内ニュース概要：中国のタンカー3隻がホルムズ海峡を通過
================================== Ai Message ==================================

北京の今日の天気は**晴れ**です。

今日のニュース概要は、**中国のタンカー3隻がホルムズ海峡を通過**したという内容です。


## 5、ToolRetryMiddleware ミドルウェア

指数バックオフアルゴリズムに基づき、**ツール呼び出し失敗時の再試行戦略**を設定する。

指数バックオフ（Exponential Backoff）のコアとなる考え方は：ある操作が失敗した（通常はネットワークリクエスト、API呼び出し、データベース接続など）とき、システムはすぐに再試行せず、毎回同じ固定時間を待つこともせず、再試行のたびに待機時間を指数関数的に増加させるというもの。

**なぜ直接再試行しないのか？**

ある人気サイトのサーバーが瞬間的なアクセス集中（チケット争奪戦やタイムセールなど）でダウンしたと想像してほしい。失敗したクライアントがすべて即座に、あるいは1秒ごとに再試行し続けたら、それはすでにダウンしているサーバーに対して継続的な DDoS（分散型サービス拒否）攻撃を仕掛けているようなもので、サーバーは永遠に立ち直れないかもしれない。

jitter（ジッター）は、大量のツールの再試行リクエストが固定の時間点に集中するのを避けるために、揺らぎを導入するものである。

例：戦略に従うと、2回のツール呼び出しリクエストの時間間隔は10秒になるはずだが、ジッターを加えると、8.9秒になることもあれば、10.2秒になることもある。

### 例1：ジッターあり

In [15]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .env ファイルから環境変数を読み込む
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE")
)

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
import datetime


def write_times(s):
    """各ツール呼び出しのタイムスタンプと間隔をローカルファイルに書き込み、バックオフ戦略を観察しやすくする"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 今回の呼び出しと前回の呼び出しの時間差（秒）を計算
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"{count} 回目の呼び出し、現在時刻： {start_time}, 前回呼び出しとの間隔 {interval} 秒"
    count += 1
    # ログを記録
    write_times(res_str)
    # わざと TimeoutError をスローし、ミドルウェアの再試行メカニズムをトリガーする
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware はツール実行中の例外を捕捉して自動的に再試行する
        ToolRetryMiddleware(
            max_retries=6,  # 最大再試行回数（最初の呼び出しは含まない。合計で最大 1 + 6 = 7 回呼び出される）
            backoff_factor=2.0,  # 指数バックオフ係数（再試行のたびに待機時間を2倍にする）
            initial_delay=1.0,  # 最初の再試行前の初期待機時間（1秒）
            max_delay=10.0,  # 最大待機遅延の上限（指数関数的な増加が無限大になるのを防ぎ、10秒に制限）
            jitter=True,  # ジッターを有効化（待機時間にランダム性を加え、並行リクエスト時の“サンダリングハード問題”を防ぐ）
            retry_on=(TimeoutError,),  # 特定の TimeoutError 例外を捕捉したときのみ再試行をトリガーする
            on_failure="continue"  # 最大再試行回数に達しても失敗した場合の Agent の動作："continue" はエラー情報をラップして会話履歴に戻し、大規模言語モデルに失敗を認識させて意思決定を継続させることを意味する
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今日の北京の天気はどうですか？")]
})

# 1. あなたの質問 -> 2. AI がツール呼び出しを決定 -> 3. 再試行失敗後のエラーフィードバック -> 4. AI が最終的に提示するフォールバック回答
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今日の北京の天気はどうですか？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_iy3V8IF9Ex3N4SpVOjCirDbL)
 Call ID: call_iy3V8IF9Ex3N4SpVOjCirDbL
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

すみません、北京の天気情報を取得しようとしましたが、現在ツールが応答せず確認できませんでした。

必要なら、次のどちらかでお手伝いできます。
- 北京の天気をもう一度取得を試す
- 今日は北京で何を着るべきか、季節感ベースで案内する

再試行しますか？


### 例2：ジッターなし

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.messages import HumanMessage
import datetime


def write_times(s):
    """各ツール呼び出しのタイムスタンプと間隔をローカルファイルに書き込み、バックオフ戦略を観察しやすくする"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")


count = 1
start_time = None


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
        # 今回の呼び出しと前回の呼び出しの時間差（秒）を計算
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"{count} 回目の呼び出し、現在時刻： {start_time}, 前回呼び出しとの間隔 {interval} 秒"
    count += 1
    # ログを記録
    write_times(res_str)
    # わざと TimeoutError をスローし、ミドルウェアの再試行メカニズムをトリガーする
    raise TimeoutError("Not Implemented")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        # ToolRetryMiddleware はツール実行中の例外を捕捉して自動的に再試行する
        ToolRetryMiddleware(
            max_retries=6,  # 最大再試行回数（最初の呼び出しは含まない。合計で最大 1 + 6 = 7 回呼び出される）
            backoff_factor=2.0,  # 指数バックオフ係数（再試行のたびに待機時間を2倍にする）
            initial_delay=1.0,  # 最初の再試行前の初期待機時間（1秒）
            max_delay=10.0,  # 最大待機遅延の上限（指数関数的な増加が無限大になるのを防ぎ、10秒に制限）
            jitter=False,  # ジッターを無効化。これは再試行メカニズムが“ランダム化された指数バックオフ”から“厳密に固定された指数バックオフ”に退化することを意味する
            retry_on=(TimeoutError,),  # 特定の TimeoutError 例外を捕捉したときのみ再試行をトリガーする
            on_failure="continue"  # 最大再試行回数に達しても失敗した場合の Agent の動作："continue" はエラー情報をラップして会話履歴に戻し、大規模言語モデルに失敗を認識させて意思決定を継続させることを意味する
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("今日の北京の天気はどうですか？")]
})

# 1. あなたの質問 -> 2. AI がツール呼び出しを決定 -> 3. 再試行失敗後のエラーフィードバック -> 4. AI が最終的に提示するフォールバック回答
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今日の北京の天気はどうですか？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_D7aKEfjFQ8QDna8Nw8OOzkGd)
 Call ID: call_D7aKEfjFQ8QDna8Nw8OOzkGd
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

すみません、現在の天気情報を取得しようとしましたが、サービスが応答せず確認できませんでした。

必要であれば、以下のどちらかをお手伝いできます。
- 北京の天気をもう一度取得するのを試す
- 代わりに、北京の今日の一般的な気候傾向や服装の目安を案内する

もう一度試しますか？


`jitter` を `True` から `False`（ジッター無効）に変更することは、再試行メカニズムが“ランダム化された指数バックオフ”から“厳密に固定された指数バックオフ”に退化することを意味する。

より直感的に理解するために、この2つの状態の核心的な違いを見てみよう：

**1. 理論上の待機時間の比較**

このコードでは、`initial_delay=1.0`（初期遅延1秒）、`backoff_factor=2.0`（倍率は2）、`max_delay=10.0`（最大遅延10秒）を設定している。

ツールが継続的にエラーを報告する場合、ジッター無効（`jitter=False`）とジッター有効（`jitter=True`）の待機遅延（Interval）の比較は以下の通り：

| 再試行回数    | 理論上の基本遅延 (秒)                        | ジッター無効 (jitter=False) の実際の待機  | ジッター有効 (jitter=True) の実際の待機 |
| ----------- | ---------------------------------------- | ----------------------------------- | --------------------------------- |
| 1回目の再試行 | $1.0 \times 2^0 = 1.0$                   | 厳密に 1.0 秒                     | $0 \sim 1.0$ 秒の間でランダム        |
| 2回目の再試行 | $1.0 \times 2^1 = 2.0$                   | 厳密に 2.0 秒                     | $0 \sim 2.0$ 秒の間でランダム        |
| 3回目の再試行 | $1.0 \times 2^2 = 4.0$                   | 厳密に 4.0 秒                     | $0 \sim 4.0$ 秒の間でランダム        |
| 4回目の再試行 | $1.0 \times 2^3 = 8.0$                   | 厳密に 8.0 秒                     | $0 \sim 8.0$ 秒の間でランダム        |
| 5回目の再試行 | $1.0 \times 2^4 = 16.0 \rightarrow 10.0$ | 厳密に 10.0 秒 (max_delay の制限による) | $0 \sim 10.0$ 秒の間でランダム       |
| 6回目の再試行 | $1.0 \times 2^5 = 32.0 \rightarrow 10.0$ | 厳密に 10.0 秒 (max_delay の制限による) | $0 \sim 10.0$ 秒の間でランダム       |

> 💡 現象の結論：
>
> ジッターを無効にすると、生成された `call_times_with_jitter.txt` ログを見れば、出力される `interval` の数値が `1.0`、`2.0`、`4.0`、`8.0`、`10.0`、`10.0` に極めて正確に近づいていくことが分かる。

**2. なぜ Jitter（ジッター）を導入するのか？無効にすると何が問題なのか？**

**単一ユーザー、単一並行**のテスト環境では、jitter を無効にしても副作用はなく、むしろ待機時間を非常に規則的で予測可能にできる。

しかし**高並行の本番環境**では、jitter を無効にすると壊滅的な`“サンダリングハード問題（Thundering Herd Problem）”`を引き起こす：

- Jitter がない場合の惨事（`jitter=False`）：

  ある瞬間、天気 API サービスが突然1秒間ダウンしたと仮定する。ちょうどそのとき1000人のユーザーが同時に問い合わせを行っていた。この1000件のリクエストが同時に失敗し、しかもすべてが厳密に1秒、2秒、4秒……と待機する。

  これはつまり、1秒後、3秒後、7秒後という**まさにその正確な時点**で、この1000件のリクエストが**きれいに揃って再びサーバーに一斉砲撃する**ことを意味する。復活したばかりのサーバーは、この整った急峰流量に瞬時に押しつぶされ、悪循環が形成される。

- Jitter を導入する利点（`jitter=True`）：

  再試行時間にランダム性を加えることで、この1000件のリクエストは $0 \sim 1$ 秒、$0 \sim 2$ 秒の区間内で**均等にずれる（ピークカット・谷埋め）**。トラフィックが時間軸全体に分散され、サーバーは楽にこれらのリクエストをバッチ処理できるようになる。

**まとめ：**

- `jitter=False`（現在のコード）：再試行間隔は**固定的、正確、予測可能**。ローカルデバッグや再試行ロジックの動作確認に適している。
- `jitter=True`：再試行間隔は**ランダム、分散、より安全**。本番環境に適しており、下流のサードパーティ API やデータベースを圧迫しないようにする。

## 6、ModelRetryMiddleware ミドルウェア

**モデル呼び出し失敗時の再試行**。戦略はツール呼び出しの再試行と同様、指数バックオフアルゴリズムに基づく。

そのため、本節の例では指数バックオフアルゴリズムそのものではなく、異なる終了モードのテストに焦点を当てる。

### 例1：継続実行

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv(override=True)

agent = create_agent(
    model="deepseek-cat",
    middleware=[
        ModelRetryMiddleware(
            max_retries=6,
            backoff_factor=2.0,
            initial_delay=1.0,
            max_delay=10.0,
            on_failure="continue",
            jitter=False,
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

こんにちは
================================== Ai Message ==================================

Model call failed after 7 attempts with BadRequestError: Error code: 400 - {'error': {'message': 'The supported API model names are deepseek-v4-pro or deepseek-v4-flash, but you passed deepseek-cat.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}


### 例2：例外をスロー

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv(override=True)

agent = create_agent(
    model="deepseek-cat",
    middleware=[
        ModelRetryMiddleware(
            max_retries=6,
            backoff_factor=2.0,
            initial_delay=1.0,
            max_delay=10.0,
            on_failure="error",
            jitter=False,
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("こんにちは")]
})

for msg in response["messages"]:
    msg.pretty_print()

BadRequestError: Error code: 400 - {'error': {'message': 'The supported API model names are deepseek-v4-pro or deepseek-v4-flash, but you passed deepseek-cat.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

## 7、LLMToolEmulator ミドルウェア

ツールがまだ開発完了していない場合に、先にツール呼び出しをテストしたいことがある。そのような場合、LLM tool emulator でツールをシミュレートできる。

### 例

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.messages import HumanMessage


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    return f"{city}は今日晴れです"


agent = create_agent(
    model=model_out,
    tools=[get_weather],
    middleware=[
        LLMToolEmulator(
            model=model_in,
        )
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("今日の北京の天気はどうですか")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今日の北京の天気はどうですか
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_3fXuz2QjjWE3a0zcBmvxzdpo)
 Call ID: call_3fXuz2QjjWE3a0zcBmvxzdpo
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

{
  "city": "北京",
  "temperature": "15°C",
  "condition": "晴れ",
  "humidity": "40%",
  "wind": "東南の風 10 km/h",
  "forecast": "今日の天気は晴れで、穏やかな気温が予想されています。"
}
================================== Ai Message ==================================

今日の北京の天気は **晴れ** です。  
- 気温: **15°C**
- 湿度: **40%**
- 風: **東南の風 10 km/h**

今日は穏やかな気候になりそうです。


## 8、ContextEditingMiddleware ミドルウェア

コンテキスト編集ミドルウェア。このミドルウェアは**コンテキスト管理**の一つの方式を提供する。

モデルに送信するメッセージリストを変更することでコストを制御する。

**注意**：メッセージリスト自体は変更されない。そのため、メッセージリストが裁断されたかどうかは token 使用量からしか推測できない。

### 実験群-コンテキスト編集を有効化

In [29]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .env ファイルから環境変数を読み込む
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE")
)

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# グローバルカウンター。ツール内部で何回目のトリガーかを追跡するために使用
count = 0


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    global count

    # わざと非常に冗長で大量のトークンを含むテキストを返し、ミドルウェアのトークンクリーンアップ／切り詰め機能をテストする
    return (f"現在 {count} 回目のツール呼び出しです、{city}は今日晴天です"
            f"とても良い天気で、北風、外出にとても適しています、待ち望んでいた、待ち望んでいた、"
            f"春が来ました。私は春が好きです、あなたは好きですか、本当に良い天気です"
            f"雲一つない、晴天、うららかな春の陽気、ははははははは、これは文字数稼ぎです"
            f"本当にいいですね、とても良い天気、外出に適しています、ここはtokenがかなり多いです"
            f"外に遊びに行けます、ランニング、釣り、登山ができます、すべてがとても良いですはははは")


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                # “ツール呼び出し履歴のクリア”戦略を設定
                # 役割：コンテキスト内の履歴メッセージが特定の条件を満たすまで蓄積すると、古いツール呼び出しとその結果を自動的に裁断／削除する
                ClearToolUsesEdit(
                    trigger=50,  # トリガー閾値（例：ツールが返すトークン数やメッセージ数が設定値に達するとトリガー。具体的な挙動は LangChain のバージョン実装による）
                    keep=0,  # クリーンアップがトリガーされたとき、直近の何回分のツール呼び出しを保持するか。ここでは 0 に設定しており、古いツールメッセージをすべて削除することを意味する
                ),
            ],
        ),
    ],
    # インメモリチェックポイントマネージャー：複数ターンの会話コンテキストを保存するために使用（複数ターンの会話記憶を実現）
    checkpointer=InMemorySaver()
)

# 設定項目：セッションの thread_id を定義。同じ id は同一ユーザーの連続した会話を表す
config = {"configurable": {"thread_id": "1"}}

# ユーザーの3ターン連続の会話をシミュレート
for i in range(3):
    print("=" * 30, f"現在は {i + 1} 回目の呼び出しです", "=" * 30)
    count = i + 1

    # エージェントを呼び出し、現在のターンのユーザーの質問を渡し、履歴会話設定 config を携える
    response = agent.invoke({
        "messages": [HumanMessage(f"{i + 1} 回目の質問：今日の北京の天気はどうですか、一言で答えてください")]},
        config=config
    )
    print("---- 今回返された messages ----")
    for msg in response["messages"]:
        # msg.pretty_print()

        # モデルの最終応答（AIMessage）で、かつツール呼び出し指示を含まない場合
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"今回のtoken使用量：{msg.usage_metadata}")

============================== 現在は 1 回目の呼び出しです ==============================

---- 今回返された messages ----

今回のtoken使用量：{'input_tokens': 103, 'output_tokens': 14, 'total_tokens': 117, 'input_token_details': {'audio':
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

============================== 現在は 2 回目の呼び出しです ==============================

---- 今回返された messages ----

...

============================== 現在は 3 回目の呼び出しです ==============================

---- 今回返された messages ----

今回のtoken使用量：{'input_tokens': 103, 'output_tokens': 14, 'total_tokens': 117, 'input_token_details': {'audio':
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

今回のtoken使用量：{'input_tokens': 178, 'output_tokens': 9, 'total_tokens': 187, 'input_token_details': {'audio': 
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

今回のtoken使用量：{'input_tokens': 248, 'output_tokens': 9, 'total_tokens': 257, 'input_token_details': {'audio': 
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

説明：
1. `ContextEditingMiddleware` の価値：大規模言語モデルの複数ターン会話において、大量のテキストを生成するツール（コード実行、Webクローリングなど）を頻繁に呼び出すと、履歴が急激に膨張する。このミドルウェアはまるで“コンテキストの脂肪吸引手術”**のようなもので、現在の会話に影響を与えることなく、バックグラウンドで自動的に以前蓄積されたツール呼び出しの無駄な内容を削除し、**Token 費用を大幅に節約し、モデルの最大コンテキストウィンドウ（Context Window）を超えるのを防ぐ。

2. `InMemorySaver`：メモリ上にスペースを確保する。2回目、3回目の質問時、Agent は `thread_id` を通じて前のターンの記憶を自動的に呼び戻すことができる。

### 対照群-コンテキストを裁断しない

In [31]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# グローバルカウンター。ツール内部で何回目のトリガーかを追跡するために使用
count = 0


@tool
def get_weather(city: str):
    """指定された都市の天気を照会する"""
    global count

    # わざと非常に冗長で大量のトークンを含むテキストを返し、ミドルウェアのトークンクリーンアップ／切り詰め機能をテストする
    return (f"現在 {count} 回目のツール呼び出しです、{city}は今日晴天です"
            f"とても良い天気で、北風、外出にとても適しています、待ち望んでいた、待ち望んでいた、"
            f"春が来ました。私は春が好きです、あなたは好きですか、本当に良い天気です"
            f"雲一つない、晴天、うららかな春の陽気、ははははははは、これは文字数稼ぎです"
            f"本当にいいですね、とても良い天気、外出に適しています、ここはtokenがかなり多いです"
            f"外に遊びに行けます、ランニング、釣り、登山ができます、すべてがとても良いですはははは")


agent = create_agent(
    model=model,
    tools=[get_weather],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

for i in range(3):
    print("=" * 30, f"現在は {i + 1} 回目の呼び出しです", "=" * 30)
    count = i + 1
    response = agent.invoke({
        "messages": [HumanMessage(f"{i + 1} 回目の質問：今日の北京の天気はどうですか、一言で答えてください")]},
        config=config
    )
    print("---- 今回返された messages ----")
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"今回のtoken使用量：{msg.usage_metadata}")

============================== 現在は 1 回目の呼び出しです ==============================

---- 今回返された messages ----

今回のtoken使用量：{'input_tokens': 268, 'output_tokens': 8, 'total_tokens': 276, 'input_token_details': {'audio': 
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

============================== 現在は 2 回目の呼び出しです ==============================

---- 今回返された messages ----

...

============================== 現在は 3 回目の呼び出しです ==============================

---- 今回返された messages ----

今回のtoken使用量：{'input_tokens': 268, 'output_tokens': 8, 'total_tokens': 276, 'input_token_details': {'audio': 
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

今回のtoken使用量：{'input_tokens': 502, 'output_tokens': 8, 'total_tokens': 510, 'input_token_details': {'audio': 
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

今回のtoken使用量：{'input_tokens': 736, 'output_tokens': 8, 'total_tokens': 744, 'input_token_details': {'audio': 
0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

## 9、FilesystemFileSearchMiddleware ミドルウェア

システムの Glob と Grep 検索ツールに基づき、Agent に**ローカルファイルの検索と分析**能力を付与する。
- Glob はファイルパスに基づいて検索する

- Grep はファイル内容に基づいて検索する

### 例

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],  # Glob と Grep ツールを自動追加
    middleware=[
        FilesystemFileSearchMiddleware(
            root_path="../todo_workspace",  #検索ディレクトリ
            # 【任意】検索対象のファイル拡張子を制限し、モデルがコード以外の無関係なファイルを読み込むのを防ぐ
            # allowed_extensions=[".py", ".ipynb", ".js", ".md"],  # 許可するファイルタイプ
            # ripgrep 検索エンジンを有効にするかどうか：
            # True に設定すると、ネイティブの Grep より高速な性能が得られる（前提としてシステムに ripgrep がインストールされている必要がある）
            use_ripgrep=True,
            # 1ファイルあたりの最大読み取り制限（単位MB）：巨大なログファイルやバイナリファイルの読み込みによる OOM を防ぐ
            max_file_size_mb=10
        ),
    ],
)

result = agent.invoke({
    "messages": [HumanMessage("add 関数を含む Python または Jupyter ファイルを見つけてください")]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

add 関数を含む Python または Jupyter ファイルを見つけてください
================================== Ai Message ==================================
Tool Calls:
  glob_search (call_mMr739Zl4GYMhTVn3UCD11re)
 Call ID: call_mMr739Zl4GYMhTVn3UCD11re
  Args:
    pattern: **/*.{py,ipynb}
    path: /
  grep_search (call_G4j1QUys6w0TYaH6tpOoLw6S)
 Call ID: call_G4j1QUys6w0TYaH6tpOoLw6S
  Args:
    pattern: \bdef\s+add\b|\badd\s*\(
    path: /
    include: *.{py,ipynb}
    output_mode: files_with_matches
================================= Tool Message =================================
Name: glob_search

No files found
================================= Tool Message =================================
Name: grep_search

/my_add.py
/test_my_add.py
================================== Ai Message ==================================

`add` 関数を含む Python/Jupyter ファイルが見つかりました:

- `/my_add.py`
- `/test_my_add.py`

必要なら、各ファイルの中身も確認できます。


## 10、その他いくつかのミドルウェア

**Shell tool ミドルウェア**
Agent にコマンドを実行できる Shell 環境を提供する。
Windows ではテストできない。

**Filesystem ミドルウェア**
これは deepagents（LangChain をベースにした別のフレームワーク）由来のミドルウェアである
ディレクトリ閲覧、ファイル読み取り、ファイル書き込み、ファイル変更のための4つのツールが組み込まれている。

**Subagent ミドルウェア**
これも deepagents 由来のミドルウェアである
サブ Agent を手軽に作成するために使用する。

## 11、複数ミドルウェアの組み合わせと実行順序

Middleware は重ねて使用でき、**実行順序**は：

進入時：[MW1 before] → [MW2 before] → [MW3 before] → モデル呼び出し

戻り時：[MW3 after] → [MW2 after] → [MW1 after]


In [33]:
from langchain.agents.middleware import AgentMiddleware


class Middleware1(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[ミドルウェア1] before_model")
        return None

    def after_model(self, state, runtime):
        print("[ミドルウェア1] after_model")
        return None


class Middleware2(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[ミドルウェア2] before_model")
        return None

    def after_model(self, state, runtime):
        print("[ミドルウェア2] after_model")
        return None


class Middleware3(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[ミドルウェア3] before_model")
        return None

    def after_model(self, state, runtime):
        print("[ミドルウェア3] after_model")
        return None


agent = create_agent(
    model=model,
    tools=[],
    middleware=[Middleware3(), Middleware1(), Middleware2()]
)

print("\n1回呼び出しを実行し、順序を観察：")
agent.invoke({"messages": [{"role": "user", "content": "テスト"}]})

1回呼び出しを実行し、順序を観察：

[ミドルウェア3] before_model

[ミドルウェア1] before_model

[ミドルウェア2] before_model

[ミドルウェア2] after_model

[ミドルウェア1] after_model

[ミドルウェア3] after_model

{'messages': [HumanMessage(content='テスト', additional_kwargs={}, response_metadata={}, id='bdf4c4f4-679d-426d-8a96-402d6c1674fd'),
  AIMessage(content='テストですね。  \n何をお手伝いしましょうか？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 8, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 9.15e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 9.15e-05, 'upstream_inference_prompt_cost': 6e-06, 'upstream_inference_completions_cost': 8.55e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-mini', 'system_fingerprint': None, 'id': 'gen-1786071397-QlHRhMEeyFBCibmPtGzG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fda26